# Colab H100 TabPFN Matbench Notebook

This notebook is a Colab-ready version of the project workflow. It is designed to run without the local Conda environment and to use a CUDA GPU runtime, ideally H100, for TabPFN inference.

Project question: can `TabPFNRegressor` improve small-data materials property prediction on selected Matbench tasks using Magpie composition features?

## Colab runtime instructions

Before running the notebook in Colab:

1. Open this notebook in Google Colab.
2. Go to **Runtime > Change runtime type**.
3. Choose **GPU** as the hardware accelerator.
4. If your Colab plan exposes GPU type selection, choose **H100** if available.
5. Run the GPU check cell below.

Important: Colab GPU type is not guaranteed. Google's Colab FAQ says GPU types and availability vary over time. If you get A100/L4/T4 instead of H100, the notebook can still run smaller tasks, but the larger `matbench_expt_gap` TabPFN run may be slower or may run out of memory.

In [ ]:
# GPU check
!nvidia-smi

import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print('GPU memory GB:', round(props.total_memory / 1024**3, 2))
else:
    raise RuntimeError('No CUDA GPU detected. In Colab, set Runtime > Change runtime type > GPU.')

## Install dependencies

Matbench 0.6 has old pinned dependency metadata, so we install the modern scientific stack first and then install `matbench==0.6` with `--no-deps`. This mirrors the local project environment fix.

In [ ]:
%pip install -q 'numpy<3' 'pandas<3' 'scikit-learn>=1.3,<2' 'scipy>=1.11' 'matminer==0.10.1' 'pymatgen>=2023' 'tabpfn==8.0.3' matplotlib seaborn joblib tqdm shap pyyaml

%pip install -q --no-deps matbench==0.6

In [ ]:
# Import check
import matbench
import matminer
import numpy as np
import pandas as pd
import sklearn
import tabpfn
import torch

print('matbench', getattr(matbench, '__version__', 'unknown'))
print('matminer', getattr(matminer, '__version__', 'unknown'))
print('sklearn', sklearn.__version__)
print('tabpfn', getattr(tabpfn, '__version__', 'unknown'))
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

## Set TabPFN token safely

Do not hard-code the token into the notebook. The cell below first tries Colab Secrets (`TABPFN_TOKEN`) and otherwise prompts you with a hidden input field.

In Colab, you can add a secret from the left sidebar: **Secrets > Add new secret > name = TABPFN_TOKEN**.

In [ ]:
import getpass
import os

try:
    from google.colab import userdata
    token = userdata.get('TABPFN_TOKEN')
except Exception:
    token = None

if not token:
    token = getpass.getpass('Paste TABPFN_TOKEN: ')

os.environ['TABPFN_TOKEN'] = token
print('TABPFN_TOKEN is set:', bool(os.environ.get('TABPFN_TOKEN')))

## Shared project code

The functions below reproduce the local project workflow:

- load Matbench task,
- extract composition from composition or structure inputs,
- compute Magpie features,
- evaluate models on official Matbench folds,
- save metrics and predictions.

In [ ]:
from __future__ import annotations

import json
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matbench.bench import MatbenchBenchmark
from matminer.featurizers.composition import ElementProperty
from pymatgen.core import Composition
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from tabpfn import TabPFNRegressor

warnings.filterwarnings('ignore', category=UserWarning)

OUTPUT_DIR = Path('/content/matbench_tabpfn_outputs')
METRICS_DIR = OUTPUT_DIR / 'metrics'
PREDICTIONS_DIR = OUTPUT_DIR / 'predictions'
FIGURES_DIR = OUTPUT_DIR / 'figures'
FEATURE_DIR = OUTPUT_DIR / 'features'
for d in [METRICS_DIR, PREDICTIONS_DIR, FIGURES_DIR, FEATURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
FEATURE_SET = 'magpie'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using TabPFN device:', DEVICE)
print('Output dir:', OUTPUT_DIR)

In [ ]:
def load_matbench_task(task_name: str):
    benchmark = MatbenchBenchmark(autoload=False, subset=[task_name])
    task = list(benchmark.tasks)[0]
    task.load()
    if task.metadata['task_type'] != 'regression':
        raise ValueError(f'Expected regression task, got {task.metadata["task_type"]!r}')
    return task


def to_composition(value) -> Composition:
    if isinstance(value, Composition):
        return value
    if hasattr(value, 'composition'):
        return value.composition
    return Composition(value)


def get_composition_inputs(task) -> pd.Series:
    input_type = task.metadata['input_type']
    if input_type == 'composition':
        return task.df[input_type]
    if input_type == 'structure':
        return task.df[input_type].map(lambda structure: structure.composition)
    raise ValueError(f'Unsupported input type: {input_type!r}')


def featurize_magpie(task_name: str, inputs: pd.Series, use_cache: bool = True) -> pd.DataFrame:
    cache_path = FEATURE_DIR / f'{task_name}_magpie_features.csv'
    if use_cache and cache_path.exists():
        features = pd.read_csv(cache_path, index_col=0)
        features.index = inputs.index
        return features

    compositions = inputs.map(to_composition)
    feature_input = pd.DataFrame({'composition': compositions}, index=inputs.index)
    featurizer = ElementProperty.from_preset('magpie')
    features = featurizer.featurize_dataframe(
        feature_input,
        col_id='composition',
        ignore_errors=False,
        inplace=False,
        pbar=True,
    )
    features = features.drop(columns=['composition'])
    features = features.apply(pd.to_numeric, errors='coerce')
    features = features.replace([np.inf, -np.inf], np.nan)
    features.to_csv(cache_path)
    return features

In [ ]:
def build_model(model_name: str, n_estimators: int):
    if model_name == 'random_forest':
        regressor = RandomForestRegressor(
            n_estimators=n_estimators,
            random_state=RANDOM_SEED,
            n_jobs=-1,
        )
    elif model_name == 'extra_trees':
        regressor = ExtraTreesRegressor(
            n_estimators=n_estimators,
            random_state=RANDOM_SEED,
            n_jobs=-1,
        )
    elif model_name == 'tabpfn':
        regressor = TabPFNRegressor(
            n_estimators=n_estimators,
            random_state=RANDOM_SEED,
            device=DEVICE,
            inference_precision='auto',
            memory_saving_mode='auto',
            show_progress_bar=False,
        )
    else:
        raise ValueError(f'Unsupported model: {model_name}')

    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', regressor),
    ])


def predict_in_batches(model: Pipeline, X_test: pd.DataFrame, batch_size: int | None):
    if batch_size is None or batch_size <= 0 or batch_size >= len(X_test):
        return model.predict(X_test)

    preprocessor = model[:-1]
    regressor = model.steps[-1][1]
    X_test_preprocessed = preprocessor.transform(X_test)
    preds = []
    for start in range(0, len(X_test_preprocessed), batch_size):
        stop = start + batch_size
        preds.append(regressor.predict(X_test_preprocessed[start:stop]))
    return np.concatenate(preds)


def evaluate_task_model(
    task_name: str,
    model_name: str,
    *,
    n_estimators: int,
    predict_batch_size: int | None = None,
):
    start_time = time.time()
    task = load_matbench_task(task_name)
    target_col = task.metadata['target']
    unit = task.metadata.get('unit')
    input_type = task.metadata['input_type']

    X_all = featurize_magpie(task_name, get_composition_inputs(task))
    y_all = task.df[target_col]

    metrics = []
    predictions = []

    for fold in task.folds_nums:
        print(f'{task_name} | {model_name} | fold {fold}')
        X_train_raw, y_train = task.get_train_and_val_data(fold)
        X_test_raw, y_test = task.get_test_data(fold, include_target=True)

        X_train = X_all.loc[X_train_raw.index]
        X_test = X_all.loc[X_test_raw.index]

        model = build_model(model_name, n_estimators)
        model.fit(X_train, y_train)
        y_pred = predict_in_batches(model, X_test, predict_batch_size)

        metrics.append({
            'task': task_name,
            'target': target_col,
            'unit': unit,
            'matbench_input_type': input_type,
            'feature_set': FEATURE_SET,
            'model': model_name,
            'fold': fold,
            'train_size': len(X_train),
            'test_size': len(X_test),
            'n_features': X_train.shape[1],
            'mae': mean_absolute_error(y_test, y_pred),
            'r2': r2_score(y_test, y_pred),
        })

        predictions.append(pd.DataFrame({
            'task': task_name,
            'model': model_name,
            'fold': fold,
            'mbid': y_test.index,
            'y_true': y_test.to_numpy(),
            'y_pred': y_pred,
            'absolute_error': np.abs(y_test.to_numpy() - y_pred),
        }))

    metrics_df = pd.DataFrame(metrics)
    predictions_df = pd.concat(predictions, ignore_index=True)
    summary = {
        'task': task_name,
        'target': target_col,
        'unit': unit,
        'matbench_input_type': input_type,
        'feature_set': FEATURE_SET,
        'model': model_name,
        'n_estimators': n_estimators,
        'predict_batch_size': predict_batch_size,
        'n_samples': int(len(task.df)),
        'n_folds': int(len(task.folds_nums)),
        'mean_mae': float(metrics_df['mae'].mean()),
        'std_mae': float(metrics_df['mae'].std(ddof=1)),
        'mean_r2': float(metrics_df['r2'].mean()),
        'elapsed_seconds': float(time.time() - start_time),
    }

    stem = f'{task_name}_{model_name}_{FEATURE_SET}'
    metrics_df.to_csv(METRICS_DIR / f'{stem}_metrics.csv', index=False)
    predictions_df.to_csv(PREDICTIONS_DIR / f'{stem}_predictions.csv', index=False)
    (METRICS_DIR / f'{stem}_summary.json').write_text(json.dumps(summary, indent=2))

    print(json.dumps(summary, indent=2))
    return metrics_df, predictions_df, summary

## Configure experiments

Recommended Colab/H100 order:

1. `matbench_steels`: sanity check and fast TabPFN reproduction.
2. `matbench_jdft2d`: smaller structure task, using composition-only proxy features.
3. `matbench_expt_gap`: larger 4604-sample task that failed locally on Apple MPS. This is the main H100 stress test.

If you did not receive an H100/A100, set `RUN_EXPT_GAP_TABPFN = False`.

In [ ]:
RUN_STEELS = True
RUN_JDFT2D = True
RUN_EXPT_GAP_TABPFN = True
RUN_CLASSICAL_BASELINES = True

# H100 should handle larger batches; lower this if you get CUDA OOM.
TABPFN_BATCH_SIZE_SMALL = 256
TABPFN_BATCH_SIZE_EXPT_GAP = 256

CLASSICAL_N_ESTIMATORS = 300
TABPFN_N_ESTIMATORS = 8

## Run steels: Random Forest / Extra Trees / TabPFN

In [ ]:
all_metric_tables = []
all_summaries = []

if RUN_STEELS:
    if RUN_CLASSICAL_BASELINES:
        for model_name in ['random_forest', 'extra_trees']:
            m, p, s = evaluate_task_model(
                'matbench_steels',
                model_name,
                n_estimators=CLASSICAL_N_ESTIMATORS,
            )
            all_metric_tables.append(m)
            all_summaries.append(s)

    m, p, s = evaluate_task_model(
        'matbench_steels',
        'tabpfn',
        n_estimators=TABPFN_N_ESTIMATORS,
        predict_batch_size=TABPFN_BATCH_SIZE_SMALL,
    )
    all_metric_tables.append(m)
    all_summaries.append(s)

## Run JDFT2D: composition-only proxy

This task is officially structure-based. Here we deliberately use only composition extracted from structure, so this result should be described as a limited proxy baseline.

In [ ]:
if RUN_JDFT2D:
    if RUN_CLASSICAL_BASELINES:
        for model_name in ['random_forest', 'extra_trees']:
            m, p, s = evaluate_task_model(
                'matbench_jdft2d',
                model_name,
                n_estimators=CLASSICAL_N_ESTIMATORS,
            )
            all_metric_tables.append(m)
            all_summaries.append(s)

    m, p, s = evaluate_task_model(
        'matbench_jdft2d',
        'tabpfn',
        n_estimators=TABPFN_N_ESTIMATORS,
        predict_batch_size=TABPFN_BATCH_SIZE_SMALL,
    )
    all_metric_tables.append(m)
    all_summaries.append(s)

## Run experimental band gap: H100 stress test

`matbench_expt_gap` failed locally on Apple MPS due to out-of-memory errors. On H100, this is the most valuable TabPFN run in this notebook.

In [ ]:
if RUN_EXPT_GAP_TABPFN:
    if RUN_CLASSICAL_BASELINES:
        # Extra Trees is enough as a strong classical reference for this task.
        m, p, s = evaluate_task_model(
            'matbench_expt_gap',
            'extra_trees',
            n_estimators=CLASSICAL_N_ESTIMATORS,
        )
        all_metric_tables.append(m)
        all_summaries.append(s)

    m, p, s = evaluate_task_model(
        'matbench_expt_gap',
        'tabpfn',
        n_estimators=TABPFN_N_ESTIMATORS,
        predict_batch_size=TABPFN_BATCH_SIZE_EXPT_GAP,
    )
    all_metric_tables.append(m)
    all_summaries.append(s)

## Summarize results

In [ ]:
all_metrics = pd.concat(all_metric_tables, ignore_index=True)
summary_df = (
    all_metrics
    .groupby(['task', 'model', 'unit', 'matbench_input_type'])
    .agg(
        mean_mae=('mae', 'mean'),
        std_mae=('mae', 'std'),
        mean_r2=('r2', 'mean'),
        n_features=('n_features', 'mean'),
    )
    .sort_values(['task', 'mean_mae'])
    .reset_index()
)

all_metrics.to_csv(METRICS_DIR / 'colab_all_fold_metrics.csv', index=False)
summary_df.to_csv(METRICS_DIR / 'colab_model_summary.csv', index=False)
summary_df

In [ ]:
for task_name, df_task in summary_df.groupby('task'):
    fig, ax = plt.subplots(figsize=(8, 4))
    plot_df = df_task.sort_values('mean_mae')
    ax.barh(plot_df['model'], plot_df['mean_mae'], xerr=plot_df['std_mae'], alpha=0.85)
    unit = plot_df['unit'].iloc[0]
    ax.set_title(f'{task_name}: model comparison')
    ax.set_xlabel(f'Mean MAE across official folds ({unit})')
    ax.invert_yaxis()
    plt.tight_layout()
    out = FIGURES_DIR / f'colab_{task_name}_model_comparison.png'
    plt.savefig(out, dpi=200)
    plt.show()
    print('Saved', out)

## Download outputs

This creates a zip file containing metrics, predictions, cached features, and plots.

In [ ]:
import shutil
zip_path = shutil.make_archive('/content/matbench_tabpfn_outputs', 'zip', OUTPUT_DIR)
print('Created:', zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as exc:
    print('Download helper unavailable:', exc)
    print('Zip path:', zip_path)

## Notes for final report

- If `matbench_expt_gap` TabPFN runs successfully on H100, it directly addresses the local MPS limitation.
- If it still runs out of memory, reduce `TABPFN_BATCH_SIZE_EXPT_GAP` and/or `TABPFN_N_ESTIMATORS`.
- For `matbench_jdft2d`, describe the result as composition-only because the official task input is structure.
- The final project should report MAE as the primary metric and R2 as secondary context.